In [4]:
import sqlite3
from pathlib import Path
from datetime import datetime

# DB ファイルのパス（data/ サブフォルダに保存する）
DB_PATH = Path("data/tech0_search.db")

def get_connection():
    """
    DB への接続を取得する。

    row_factory を設定することで、行データを辞書のように扱える。
    data/ フォルダが存在しない場合は自動で作成する。
    """
    DB_PATH.parent.mkdir(exist_ok=True)   # data/ フォルダがなければ作る
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row        # 行データを辞書のように扱う
    return conn

def init_db():
    """
    schema.sql を読み込んで DB を初期化する。

    CREATE TABLE IF NOT EXISTS を使っているので、
    すでにテーブルが存在する場合は何もしない。
    """
    conn = get_connection()
    with open("schema.sql", "r", encoding="utf-8") as f:
        conn.executescript(f.read())    # SQL ファイルをまとめて実行する
    conn.commit()
    conn.close()
    print("DB を初期化しました（data/tech0_search.db）")

# 動作確認
init_db()

DB を初期化しました（data/tech0_search.db）


In [6]:
def _keywords_to_text(keywords) -> str:
    """keywords（リスト or 文字列）を DB 保存用のカンマ区切り文字列にする"""
    if not keywords:
        return ""
    if isinstance(keywords, str):
        return keywords
    return ",".join(keywords)


def insert_page(page: dict) -> int:
    """
    ページ情報を DB に登録する。

    INSERT OR REPLACE：同じ URL のデータがあれば上書き、なければ新規追加する。
    これにより「同じページを再クロールしたときに最新データに更新できる」。

    Args:
        page: ページ情報の辞書（crawl_url() の返り値と同形式）

    Returns:
        登録された行の id
    """
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT OR REPLACE INTO pages
            (url, title, description, full_text, author, category, keywords, word_count, crawled_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        page["url"],
        page["title"],
        page.get("description", ""),
        page.get("full_text", ""),
        page.get("author", ""),
        page.get("category", ""),
        _keywords_to_text(page.get("keywords")),   # リスト → "DX,営業店" にして保存
        page.get("word_count", 0),
        page.get("crawled_at", datetime.now().isoformat()),
    ))

    page_id = cursor.lastrowid    # 登録された行の id を取得する
    conn.commit()
    conn.close()
    return page_id


def get_all_pages() -> list:
    """全ページを登録日時の新しい順で取得する。"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM pages ORDER BY created_at DESC")
    rows = cursor.fetchall()
    conn.close()
    pages = []
    for row in rows:
        pg = dict(row)
        # 保存時にカンマ区切りにした keywords をリストに戻す
        pg["keywords"] = [k.strip() for k in (pg.get("keywords") or "").split(",") if k.strip()]
        pages.append(pg)
    return pages    # sqlite3.Row を辞書に変換して返す


# 動作確認：テストデータを登録する
test_pages = [
    {"url": "https://example.com/dx-report",
     "title": "DX戦略レポート2024",
     "description": "営業店のDX推進について解説した報告書",
     "full_text": "DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例",
     "keywords": "DX,営業店,デジタル変革",
     "word_count": 150,
     "crawled_at": "2024-01-15T10:00:00"},
    {"url": "https://example.com/ai-guide",
     "title": "AI活用ガイド",
     "description": "業務効率化のためのAI導入ガイド",
     "full_text": "AI 活用 で 業務 効率 化 を 進める 機械 学習 ツール",
     "keywords": "AI,機械学習,業務効率化",
     "word_count": 120,
     "crawled_at": "2024-02-20T15:00:00"},
]

for p in test_pages:
    pid = insert_page(p)
    print(f"登録完了：id={pid}  {p['title']}")

# 登録されたデータを確認する
all_pages = get_all_pages()
print(f"\n現在の登録ページ数：{len(all_pages)} 件")


def log_search(query: str, results_count: int, user_id: str = None) -> int:
    """
    検索ログを記録する（W6 の発展課題で実装予定）
    現在はスタブ（空の関数）として定義しています。
    W6 の発展課題で実際の記録処理を実装します。
    """
    pass  # W6 の発展課題で実装します

登録完了：id=5  DX戦略レポート2024
登録完了：id=6  AI活用ガイド

現在の登録ページ数：2 件


In [7]:
# 移行前の件数を記録しておく
before = len(get_all_pages())

# まったく同じJSONを、もう一度移行してみる
migrate_from_json("pages_w2.json")

after = len(get_all_pages())

print(f"1回目の移行後: {before} 件")
print(f"2回目の移行後: {after} 件")
print()

if before == after:
    print("✅ 件数が増えていない ＝ 同じURLは新規追加ではなく【上書き】された")
    print("   W2（JSONにappend）なら、同じデータが2セットに増えていました。")
else:
    print(f"❌ {after - before} 件増えてしまった（UNIQUE制約を確認しよう）")

NameError: name 'migrate_from_json' is not defined

In [10]:
import json

with open("pages_w2.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

print("JSONの件数:", len(pages))

urls = [p["url"] for p in pages]

print("JSONの件数:", len(urls))
print("URLの種類:", len(set(urls)))

JSONの件数: 29
JSONの件数: 29
URLの種類: 29


In [11]:
pages_db = get_all_pages()
print("DBの件数:", len(pages_db))

DBの件数: 2


In [ ]:
print("JSON:", len(pages))

JSON: 29


In [13]:
n = migrate_from_json("pages_w2.json")
print("移行した件数:", n)

NameError: name 'migrate_from_json' is not defined

In [16]:
from database import migrate_from_json

In [17]:
n = migrate_from_json("pages_w2.json")
print("移行した件数:", n)
print("DBの件数:", len(get_all_pages()))

移行した件数: 29
DBの件数: 31


In [18]:
pages = get_all_pages()
print(f"DBから読み込んだページ数：{len(pages)} 件")
if pages:
    print("先頭ページ:", pages[0]["title"])

DBから読み込んだページ数：31 件
先頭ページ: 営業店DX推進レポート


In [19]:
# まずは1ページ分のデータを使って、検索対象文字列を作ってみる
page = pages[0]

# title, description, full_text, keywords をまとめて1つの文字列にする
search_text = " ".join([
    page.get("title", ""),
    page.get("description", ""),
    page.get("full_text", ""),
    " ".join(page.get("keywords", [])),
])

print(search_text[:500])

営業店DX推進レポート 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録  DX 営業店 業務効率


In [20]:
# 検索キーワード
query = "DX"

# 大文字小文字の違いを吸収するために lower() にして比較する
found = query.lower() in search_text.lower()

print("検索キーワード:", query)
print("含まれているか:", found)

検索キーワード: DX
含まれているか: True


In [21]:
# 検索キーワードの出現回数を数える
count = search_text.lower().count(query.lower())

print("検索キーワード:", query)
print("出現回数:", count)

検索キーワード: DX
出現回数: 3


In [24]:
# 検索キーワード
query = "Scrape"

# 大文字小文字の違いを吸収するために lower() にして比較する
found = query.lower() in search_text.lower()

print("検索キーワード:", query)
print("含まれているか:", found)

検索キーワード: Scrape
含まれているか: False


In [25]:
# 検索キーワード
query = "Scrape"

# 大文字小文字の違いを吸収するために lower() にして比較する
found = query.lower() in search_text.lower()

print("検索キーワード:", query)
print("含まれているか:", found)

検索キーワード: Scrape
含まれているか: False


In [26]:
query = "DX"
results = []

# すべてのページを1件ずつ確認する
for page in pages:
    
    # そのページの検索対象文字列を作る
    text = " ".join([
        page.get("title", ""),
        page.get("description", ""),
        page.get("full_text", ""),
        " ".join(page.get("keywords", [])),
    ]).lower()

    # キーワードの出現回数を数える
    count = text.count(query.lower())

    # 1回以上出てきたページだけ結果に追加
    if count > 0:
        result = page.copy()              # 元データをコピー
        result["match_count"] = count     # 検索結果用に match_count を追加
        results.append(result)

print("検索結果件数:", len(results))
results[:2]

検索結果件数: 7


[{'id': 7,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-27T21:14:01.669731',
  'created_at': '2026-09-27 12:14:01',
  'updated_at': '2026-09-27 12:14:01',
  'match_count': 3},
 {'id': 8,
  'url': 'https://kb.techzeron-bank.example/ai-screening',
  'title': '住宅ローン審査AI導入の検討',
  'description': 'AI事前審査で融資業務をDX化する企画提案',
  'full_text': '',
  'author': '鈴木花子',
  'category': '企画',
  'keywords': ['AI', '融資', '審査'],
  'word_count': 0,
  'crawled_at': '2026-09-27T21:14:01.684416',
  'created_at': '2026-09-27 12:14:01',
  'updated_at': '2026-09-27 12:14:01',
  'match_count': 1}]

In [27]:
# match_count の大きい順に並べる
sorted_results = sorted(
    results,
    key=lambda x: x["match_count"],
    reverse=True
)

print("並べ替え後の上位3件:")
for r in sorted_results[:3]:
    print(r["title"], "->", r["match_count"])

並べ替え後の上位3件:
DX戦略レポート2024 -> 4
営業店DX推進レポート -> 3
AIチャットボットによる照会対応DX -> 3


In [28]:
def make_preview(text: str, query: str, ctx: int = 80) -> str:
    """
    キーワードが見つかった位置の前後を切り出して preview を作る

    Args:
        text: 本文
        query: 検索キーワード
        ctx: 前後に何文字表示するか

    Returns:
        preview文字列
    """
    # キーワードの位置を探す
    pos = text.lower().find(query.lower())

    # 見つからない場合は先頭200文字を返す
    if pos == -1:
        return text[:200]

    # 前後の範囲を決める
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    # 前後を切り出して返す
    return "..." + text[start:end] + "..."

In [29]:
query = "DX"
results = []

for page in pages:
    
    # 検索対象文字列を作る
    text = " ".join([
        page.get("title", ""),
        page.get("description", ""),
        page.get("full_text", ""),
        " ".join(page.get("keywords", [])),
    ]).lower()

    # 出現回数を数える
    count = text.count(query.lower())

    # ヒットしたページだけ結果に追加
    if count > 0:
        result = page.copy()
        result["match_count"] = count
        
        # previewは full_text があればそれを使い、なければ description を使う
        result["preview"] = make_preview(
            page.get("full_text") or page.get("description", ""),
            query
        )
        
        results.append(result)

# match_count順に並べる
results = sorted(results, key=lambda x: x["match_count"], reverse=True)

print("検索結果件数:", len(results))
results[:2]

検索結果件数: 7


[{'id': 5,
  'url': 'https://example.com/dx-report',
  'title': 'DX戦略レポート2024',
  'description': '営業店のDX推進について解説した報告書',
  'full_text': 'DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例',
  'author': '',
  'category': '',
  'keywords': ['DX', '営業店', 'デジタル変革'],
  'word_count': 150,
  'crawled_at': '2024-01-15T10:00:00',
  'created_at': '2026-09-27 12:05:33',
  'updated_at': '2026-09-27 12:05:33',
  'match_count': 4,
  'preview': '...DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例...'},
 {'id': 7,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-27T21:14:01.669731',
  'created_at': '2026-09-27 12:14:01',
  'updated_at': '2026-09-27 12:14:01',
  'match_count': 3,
  'preview': '...窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録...'}]

In [30]:
def _make_preview(text: str, query: str, ctx: int = 80) -> str:
    """
    マッチ箇所周辺のプレビューを生成する。

    Args:
        text: 本文
        query: 検索キーワード
        ctx: 前後に表示する文字数

    Returns:
        preview文字列
    """

    # text または query が空なら空文字を返す
    if not text or not query:
        return ""

    # キーワードの位置を探す
    pos = text.lower().find(query.lower())

    # 見つからない場合は先頭200文字を返す
    if pos == -1:
        return (text[:200] + "...") if len(text) > 200 else text

    # 前後の切り出し範囲
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    preview = ""

    if start > 0:
        preview += "..."

    preview += text[start:end]

    if end < len(text):
        preview += "..."

    return preview


def search_fulltext(query: str, pages: list) -> list:
    """
    全文検索（本文含む）を実行し、マッチ数でスコアリングする。

    Args:
        query: 検索キーワード
        pages: ページリスト

    Returns:
        マッチしたページのリスト（スコア降順）
    """

    # 空検索なら空リストを返す
    if not query.strip():
        return []

    results = []

    # 検索キーワードを小文字化
    q = query.lower()

    # 全ページを順番に確認
    for page in pages:

        # 検索対象テキストを作る
        text = " ".join([
            page.get("title", ""),
            page.get("description", ""),
            page.get("full_text", ""),
            " ".join(page.get("keywords", [])),
        ]).lower()

        # 出現回数を数える
        count = text.count(q)

        # ヒットしたページだけ追加
        if count > 0:

            r = page.copy()

            # match_count を追加
            r["match_count"] = count

            # preview を追加
            r["preview"] = _make_preview(
                page.get("full_text") or page.get("description", ""),
                query
            )

            results.append(r)

    # match_count順に並べる
    results.sort(key=lambda x: x["match_count"], reverse=True)

    return results

In [31]:
# 検索キーワード
query = "DX"

# search_fulltext を実行
results = search_fulltext(query, pages)

# 検索結果件数を表示
print("検索結果件数:", len(results))

# 上位5件を表示
for r in results[:5]:
    print("----")
    # タイトル
    print("タイトル:", r["title"])
    # キーワード出現回数
    print("match_count:", r["match_count"])
    # preview（本文の一部）
    print("preview:", r["preview"][:120])

検索結果件数: 7
----
タイトル: DX戦略レポート2024
match_count: 4
preview: DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例
----
タイトル: 営業店DX推進レポート
match_count: 3
preview: 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録
----
タイトル: AIチャットボットによる照会対応DX
match_count: 3
preview: コールセンターにAIチャットボットを導入し照会対応を効率化したDX事例
----
タイトル: 全店DX推進ロードマップ2026
match_count: 3
preview: 勘定系刷新とデジタルチャネル強化を柱とする全店DX計画の概要
----
タイトル: 営業店ペーパーレス化 全店展開レポート（2026年度上期）
match_count: 2
preview: ... パーセント 削減 した 紙 の 稟議 書 を 電子 化 する こと で 決裁 までの 日数 も 短縮 され た 今後 は 融資 部門 への 展開 を 進める DX 推進 室 は 効果 測定 を 継続 する
